# Evaluación múltiple — pipeline REDUCIDO en lote (sin PnLCalib)

Procesa **20 clips** del test de SoccerNet para evaluar **detección, tracking y clasificación de equipos** (más balón). NO ejecuta PnLCalib ni proyección a campo (esas métricas usan bounding boxes en píxeles, no posiciones de campo), por lo que cada clip tarda ~5 min en vez de ~17.

Genera un JSON reducido por clip: `integrado_eval_SNMOT-XXX.json` con bbox, id, rol (jugador/arbitro/balon) y equipo. **Sin rol portero** (requeriría calibración) y **sin posiciones de campo**.

**Rigor:** los 20 clips se eligen aleatoriamente con semilla fija (reproducible). Todos se procesan con los MISMOS parámetros (sin ajustar por clip) y se evalúan en crudo. El balón se evalúa por distancia de centros (objeto pequeño). El mapeo equipo→lado para la clasificación se hace por posición en píxeles (no campo).

In [1]:
# ---- CELDA 1: Setup + elegir los clips NO usados (re-validación) + preparar ----
import random
import numpy as np, cv2
from pathlib import Path
from collections import defaultdict, Counter

REPO_ROOT = Path("../..").resolve()
SOCCERNET_DIR = REPO_ROOT / "data" / "soccernet"
TEST_DIR = SOCCERNET_DIR / "tracking" / "test"
MODELO_DETECTOR = REPO_ROOT / "models" / "best_v5.pt"
OUT_TRACKING = REPO_ROOT / "data" / "tracking_revalidacion"   # carpeta NUEVA: no pisa los 20 anteriores
OUT_TRACKING.mkdir(parents=True, exist_ok=True)

clips_todos = sorted([d for d in TEST_DIR.iterdir() if d.is_dir() and d.name.startswith("SNMOT")])

# Los 20 clips YA evaluados (selección original con seed 2024) -> se excluyen
CLIPS_YA_USADOS = {
    "SNMOT-120","SNMOT-127","SNMOT-128","SNMOT-129","SNMOT-131","SNMOT-132",
    "SNMOT-135","SNMOT-137","SNMOT-138","SNMOT-139","SNMOT-140","SNMOT-142",
    "SNMOT-146","SNMOT-147","SNMOT-149","SNMOT-150","SNMOT-189","SNMOT-194",
    "SNMOT-196","SNMOT-198",
}

# Evaluar ahora SOLO los clips que NO se usaron antes
clips_eval = sorted([c for c in clips_todos if c.name not in CLIPS_YA_USADOS], key=lambda c: c.name)

# (opcional) limitar cuántos usar; deja None para usar TODOS los no usados
LIMITE = None
if LIMITE is not None:
    random.seed(7)   # semilla distinta de la original
    clips_eval = sorted(random.sample(clips_eval, min(LIMITE, len(clips_eval))), key=lambda c: c.name)

print(f"Clips de test totales: {len(clips_todos)}")
print(f"Excluidos (ya usados): {len(CLIPS_YA_USADOS)}")
print(f"Clips NUEVOS para re-validación: {len(clips_eval)}")
for c in clips_eval: print(f"  {c.name}")

W_IMG, H_IMG, FPS = 1920, 1080, 25

Clips de test totales: 49
Excluidos (ya usados): 20
Clips NUEVOS para re-validación: 29
  SNMOT-116
  SNMOT-117
  SNMOT-118
  SNMOT-119
  SNMOT-121
  SNMOT-122
  SNMOT-123
  SNMOT-124
  SNMOT-125
  SNMOT-126
  SNMOT-130
  SNMOT-133
  SNMOT-134
  SNMOT-136
  SNMOT-141
  SNMOT-143
  SNMOT-144
  SNMOT-145
  SNMOT-148
  SNMOT-187
  SNMOT-188
  SNMOT-190
  SNMOT-191
  SNMOT-192
  SNMOT-193
  SNMOT-195
  SNMOT-197
  SNMOT-199
  SNMOT-200


In [2]:
# ---- CELDA 2: función que procesa UN clip (tracking+cosido+balón+equipos+árbitro) ----
from ultralytics import YOLO

detector       = YOLO(str(MODELO_DETECTOR))   # SOLO tracking  (.track)
detector_balon = YOLO(str(MODELO_DETECTOR))   # instancia SEPARADA, SOLO balón (.predict)
print(f"Detector cargado. Clases: {detector.names}")

# Config del tracker (mismo que el pipeline principal, new_track_thresh 0.6)
tracker_cfg = REPO_ROOT / "configs" / "botsort_custom.yaml"
tracker_cfg.parent.mkdir(parents=True, exist_ok=True)
if not tracker_cfg.exists():
    tracker_cfg.write_text("""tracker_type: botsort
track_high_thresh: 0.25
track_low_thresh: 0.1
new_track_thresh: 0.6
track_buffer: 120
match_thresh: 0.8
fuse_score: true
gmc_method: sparseOptFlow
proximity_thresh: 0.5
appearance_thresh: 0.25
with_reid: true
model: auto
""", encoding="utf-8")

def montar_video(clip):
    """Monta el .mp4 del clip desde sus frames si no existe."""
    out = REPO_ROOT / "data" / "raw" / f"soccernet_{clip.name}.mp4"
    if out.exists(): return out
    jpgs = sorted((clip/"img1").glob("*.jpg"))
    prim = cv2.imread(str(jpgs[0])); h,w = prim.shape[:2]
    wr = cv2.VideoWriter(str(out), cv2.VideoWriter_fourcc(*"mp4v"), FPS, (w,h))
    for f in jpgs: wr.write(cv2.imread(str(f)))
    wr.release()
    return out

def iou(b1, b2):
    x1,y1,x2,y2=b1; X1,Y1,X2,Y2=b2
    xi1,yi1=max(x1,X1),max(y1,Y1); xi2,yi2=min(x2,X2),min(y2,Y2)
    if xi2<xi1 or yi2<yi1: return 0
    inter=(xi2-xi1)*(yi2-yi1); a1=(x2-x1)*(y2-y1); a2=(X2-X1)*(Y2-Y1)
    return inter/(a1+a2-inter)

print("Funciones listas.")

Detector cargado. Clases: {0: 'ball', 1: 'player', 2: 'referee'}
Funciones listas.


In [3]:
# ---- CELDA 3: tracking+cosido como función (copiado del pipeline, parametrizado) ----
def procesar_tracking(VIDEO_IN, N_FRAMES_MAX=10000):
    frames_dets = []
    results = detector.track(source=str(VIDEO_IN), stream=True, persist=False,
                             tracker=str(tracker_cfg), conf=0.25, imgsz=1280, verbose=False)
    for n, rd in enumerate(results):
        dets = []
        if rd.boxes is not None and rd.boxes.id is not None:
            for (x1,y1,x2,y2), tid, c in zip(rd.boxes.xyxy.cpu().numpy(),
                                             rd.boxes.id.cpu().numpy().astype(int),
                                             rd.boxes.cls.cpu().numpy().astype(int)):
                dets.append((int(tid), float(x1), float(y1), float(x2), float(y2), int(c)))
        frames_dets.append(dets)
    N_FRAMES = len(frames_dets)

    tracks_full = defaultdict(list)
    for f_idx, dets in enumerate(frames_dets):
        for (tid,x1,y1,x2,y2,c) in dets:
            tracks_full[tid].append((f_idx,(x1+x2)/2,y2,int(c)))

    MIN_SEG=15
    tracks2, clase2, mapa_det = {}, {}, {}
    next_id=1
    for tid,pts in tracks_full.items():
        segs,actual,pendiente,c_actual=[],[],[],None
        for p in pts:
            c=p[3]
            if c_actual is None: c_actual=c
            if c==c_actual:
                if pendiente: actual.extend([(f,x,y,c_actual) for (f,x,y,_) in pendiente]); pendiente=[]
                actual.append(p)
            else:
                pendiente.append(p)
                if len(pendiente)>=MIN_SEG:
                    if actual: segs.append((c_actual,actual))
                    c_actual,actual,pendiente=c,[(f,x,y,c) for (f,x,y,_) in pendiente],[]
        if pendiente: actual.extend([(f,x,y,c_actual) for (f,x,y,_) in pendiente])
        if actual: segs.append((c_actual,actual))
        for c_seg,seg in segs:
            nid=next_id; next_id+=1
            tracks2[nid]=[(f,x,y) for (f,x,y,c) in seg]; clase2[nid]=c_seg
            for (f,x,y,c) in seg: mapa_det[(f,tid)]=nid

    MAX_GAP,MAX_SOLAPE,RADIO_BASE,RADIO_POR_FRAME,RADIO_SOLAPE=75,12,50,5,35
    resumen,pos_pf={},{}
    for nid,pts in tracks2.items():
        if clase2[nid] not in (1,2) or len(pts)<3: continue
        arr=np.array(pts); ult=arr[-5:]; dt=ult[-1,0]-ult[0,0]
        vel=(ult[-1,1:]-ult[0,1:])/dt if dt>0 else np.zeros(2)
        resumen[nid]={"ini":int(arr[0,0]),"fin":int(arr[-1,0]),"pos_ini":arr[0,1:],"pos_fin":arr[-1,1:],"vel":vel}
        pos_pf[nid]={int(f):np.array([x,y]) for f,x,y in pts}
    cands=[]
    for ta,ra in resumen.items():
        for tb,rb in resumen.items():
            if ta==tb or clase2[ta]!=clase2[tb]: continue
            gap=rb["ini"]-ra["fin"]
            if gap>MAX_GAP or gap<-MAX_SOLAPE: continue
            if gap>0:
                d=np.linalg.norm(rb["pos_ini"]-(ra["pos_fin"]+ra["vel"]*gap)); r=RADIO_BASE+RADIO_POR_FRAME*gap
            else:
                pa=pos_pf[ta].get(rb["ini"])
                if pa is None: continue
                d,r=np.linalg.norm(rb["pos_ini"]-pa),RADIO_SOLAPE
            if d<=r: cands.append((d,ta,tb))
    cands.sort()
    uf,ui,fusion=set(),set(),{}
    for d,ta,tb in cands:
        if ta in uf or tb in ui: continue
        fusion[tb]=ta; uf.add(ta); ui.add(tb)
    def raiz(t):
        cadena=[]; vistos=set()
        while t in fusion and t not in vistos:
            cadena.append(t); vistos.add(t); t=fusion[t]
        if t in vistos:                       # ciclo: todos sus miembros al menor representante
            return min(cadena[cadena.index(t):])
        return t
    mapa_final={nid:raiz(nid) for nid in resumen}

    MIN_DET_SEG=5
    rango_seg={nid:(int(np.array(pts)[0,0]),int(np.array(pts)[-1,0])) for nid,pts in tracks2.items() if len(pts)>0}
    raiz_de={nid:int(mapa_final.get(nid,nid)) for nid in tracks2}
    espurios=set()
    for nid,pts in tracks2.items():
        if len(pts)>=MIN_DET_SEG or nid not in rango_seg: continue
        a0,a1=rango_seg[nid]; cls_a=clase2[nid]
        for otro,pts2 in tracks2.items():
            if otro==nid or clase2[otro]!=cls_a or len(pts2)<MIN_DET_SEG: continue
            if raiz_de[otro]==raiz_de[nid]: continue
            b0,b1=rango_seg[otro]
            if not (a1<b0 or a0>b1): espurios.add(nid); break

    frames_track=[]
    for f_idx,dets in enumerate(frames_dets):
        fr=[]
        for (tid,x1,y1,x2,y2,c) in dets:
            if c in (1,2):
                nid=mapa_det.get((f_idx,tid),tid)
                if nid in espurios: continue
                idf=int(mapa_final.get(nid,nid))
            else:
                idf=int(tid)
            fr.append({"id":idf,"bbox":[x1,y1,x2,y2],"cls":c})
        frames_track.append(fr)
    return frames_track, N_FRAMES

print("procesar_tracking lista.")

procesar_tracking lista.


In [4]:
purgados = 0
for f in OUT_TRACKING.glob("integrado_eval_SNMOT-*.json"):
    f.unlink()
    purgados += 1
print(f"JSON viejos purgados: {purgados}")

JSON viejos purgados: 20


In [4]:
# ---- CELDA 4: balón + equipos + árbitro + BUCLE sobre los 20 clips ----
import json

def detectar_balon(VIDEO_IN, frames_track, N_FRAMES):
    CONF_BALON=0.30; MAX_SALTO=250
    detec={}
    cap=cv2.VideoCapture(str(VIDEO_IN)); f_idx=0
    while True:
        ret,frame=cap.read()
        if not ret: break
        res=detector_balon(frame,conf=CONF_BALON,imgsz=1280,verbose=False,classes=[0])[0]   # <-- detector_balon (instancia separada)
        if res.boxes is not None and len(res.boxes)>0:
            confs=res.boxes.conf.cpu().numpy(); boxes=res.boxes.xyxy.cpu().numpy()
            k=int(np.argmax(confs)); x1,y1,x2,y2=boxes[k]
            detec[f_idx]=((x1+x2)/2,y2,float(confs[k]))
        f_idx+=1
    cap.release()
    fs=sorted(detec); desc=set()
    for i in range(1,len(fs)):
        fa,fb=fs[i-1],fs[i]
        if fa in desc:
            j=i-1
            while j>=0 and fs[j] in desc: j-=1
            if j<0: continue
            fa=fs[j]
        if fb-fa>3: continue
        xa,ya,ca=detec[fa]; xb,yb,cb=detec[fb]
        if np.hypot(xb-xa,yb-ya)/(fb-fa)>MAX_SALTO: desc.add(fa if ca<cb else fb)
    for f in desc: detec.pop(f,None)
    ID_BALON=-100
    for f_idx in range(N_FRAMES):
        frames_track[f_idx]=[d for d in frames_track[f_idx] if d["cls"]!=0]
        if f_idx in detec:
            xc,yp,_=detec[f_idx]
            frames_track[f_idx].append({"id":ID_BALON,"bbox":[xc-8,yp-16,xc+8,yp],"cls":0})
    return frames_track

def color_torso(frame,bbox):
    x1,y1,x2,y2=[int(v) for v in bbox]; hb,wb=y2-y1,x2-x1
    if hb<10 or wb<4: return None
    crop=frame[max(y1+int(0.22*hb),0):y1+int(0.50*hb),max(x1+int(0.25*wb),0):x2-int(0.25*wb)]
    if crop.size==0: return None
    hsv=cv2.cvtColor(crop,cv2.COLOR_BGR2HSV)
    pix=crop[cv2.inRange(hsv,(35,40,40),(85,255,255))==0]
    if len(pix)<30: return None
    return np.median(pix.astype(float),axis=0)

def clasificar_equipos(VIDEO_IN, frames_track):
    col_por_id=defaultdict(list)
    cap=cv2.VideoCapture(str(VIDEO_IN)); f_idx=0
    while True:
        ret,frame=cap.read()
        if not ret: break
        for d in frames_track[f_idx]:
            if d["cls"]!=1: continue
            c=color_torso(frame,d["bbox"])
            if c is not None: col_por_id[d["id"]].append(c)
        f_idx+=1
    cap.release()
    ids_val=[i for i,cs in col_por_id.items() if len(cs)>=3]
    if len(ids_val)<2: return {}, {0:(128,128,128),1:(128,128,128)}
    col_id=np.array([np.median(col_por_id[i],axis=0) for i in ids_val])
    crit=(cv2.TERM_CRITERIA_EPS+cv2.TERM_CRITERIA_MAX_ITER,100,1e-4)
    _,labels_k,centers=cv2.kmeans(col_id.astype(np.float32),2,None,crit,20,cv2.KMEANS_PP_CENTERS)
    labels_k=labels_k.ravel()
    equipo_de_id={ids_val[j]:int(labels_k[j]) for j in range(len(ids_val))}
    def realzar(c):
        c=np.array(c,dtype=float); return tuple(int(v) for v in (c/(c.max()+1e-6)*255))
    color_eq={k:realzar(centers[k]) for k in (0,1)}
    return equipo_de_id, color_eq

def asignar_roles(frames_track, equipo_de_id):
    # SIN campo: árbitro por votos cls=2 (>cls=1 y >=3); resto jugador. SIN portero.
    cls_por=defaultdict(lambda:{1:0,2:0})
    for fr in frames_track:
        for d in fr:
            if d["cls"] in (1,2): cls_por[d["id"]][d["cls"]]+=1
    rol_de,equipo_seg={},{}
    for nid in set(d["id"] for fr in frames_track for d in fr if d["cls"] in (1,2)):
        c=cls_por[nid]
        if c[2]>c[1] and c[2]>=3:
            rol_de[nid]="arbitro"; equipo_seg[nid]=-1
        else:
            rol_de[nid]="jugador"; equipo_seg[nid]=equipo_de_id.get(nid,-1)
    return rol_de, equipo_seg

# ===== BUCLE PRINCIPAL sobre los 20 clips =====
import time
for ci, clip in enumerate(clips_eval):
    t0=time.time()
    out_json = OUT_TRACKING / f"integrado_eval_{clip.name}.json"
    if out_json.exists():
        print(f"[{ci+1}/{len(clips_eval)}] {clip.name}: ya procesado, saltando")
        continue
    print(f"[{ci+1}/{len(clips_eval)}] {clip.name}: procesando...", end=" ", flush=True)
    VIDEO_IN = montar_video(clip)
    frames_track, N_FRAMES = procesar_tracking(VIDEO_IN)
    frames_track = detectar_balon(VIDEO_IN, frames_track, N_FRAMES)
    equipo_de_id, color_eq = clasificar_equipos(VIDEO_IN, frames_track)
    rol_de, equipo_seg = asignar_roles(frames_track, equipo_de_id)

    # Construir JSON reducido
    salida={"video":f"soccernet_{clip.name}.mp4","n_frames":int(N_FRAMES),
            "color_equipo_bgr":{str(k):[int(x) for x in v] for k,v in color_eq.items()},
            "frames":[]}
    for f_idx in range(N_FRAMES):
        df=[]
        for d in frames_track[f_idx]:
            nid=int(d["id"])
            if d["cls"]==0:
                df.append({"id":nid,"rol":"balon","equipo":-1,"bbox":[float(v) for v in d["bbox"]]})
            else:
                rol=rol_de.get(d["id"],"jugador")
                df.append({"id":nid,"rol":rol,"equipo":int(equipo_seg.get(d["id"],-1)),
                           "bbox":[float(v) for v in d["bbox"]]})
        salida["frames"].append(df)
    with open(out_json,"w",encoding="utf-8") as fh:
        json.dump(salida,fh)
    print(f"OK ({time.time()-t0:.0f}s, {N_FRAMES} frames)")

print("\n=== Procesamiento de los 20 clips completado ===")


[1/29] SNMOT-116: procesando... OK (117s, 750 frames)
[2/29] SNMOT-117: procesando... OK (113s, 750 frames)
[3/29] SNMOT-118: procesando... OK (116s, 750 frames)
[4/29] SNMOT-119: procesando... OK (97s, 750 frames)
[5/29] SNMOT-121: procesando... OK (125s, 750 frames)
[6/29] SNMOT-122: procesando... OK (119s, 750 frames)
[7/29] SNMOT-123: procesando... OK (115s, 750 frames)
[8/29] SNMOT-124: procesando... OK (117s, 750 frames)
[9/29] SNMOT-125: procesando... OK (117s, 750 frames)
[10/29] SNMOT-126: procesando... OK (118s, 750 frames)
[11/29] SNMOT-130: procesando... OK (120s, 750 frames)
[12/29] SNMOT-133: procesando... OK (114s, 750 frames)
[13/29] SNMOT-134: procesando... OK (114s, 750 frames)
[14/29] SNMOT-136: procesando... OK (118s, 750 frames)
[15/29] SNMOT-141: procesando... OK (118s, 750 frames)
[16/29] SNMOT-143: procesando... OK (117s, 750 frames)
[17/29] SNMOT-144: procesando... OK (118s, 750 frames)
[18/29] SNMOT-145: procesando... OK (113s, 750 frames)
[19/29] SNMOT-148: p